# StreaMAX Fit — GPU Nested Sampling (BlackJAX)
Run on Colab with A100 GPU runtime.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Set paths — edit these to match your Drive layout
REPO_DIR = '/content/drive/MyDrive/Vasily/StreaMAX_fit'
CSV_PATH = '/content/drive/MyDrive/Vasily/StreaMAX_fit/Au26_stream_4664.csv'

import os
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

In [ ]:
# 3. Install dependencies (BlackJAX from Will Handley's fork)
!pip install -q streamax corner matplotlib pandas numpy scipy tqdm
!pip install -q git+https://github.com/handley-lab/blackjax.git@nested_sampling

In [ ]:
# 4. Check GPU
import jax
print(f'JAX devices: {jax.devices()}')
assert any('gpu' in str(d).lower() or 'cuda' in str(d).lower() for d in jax.devices()), \
    'No GPU found! Change runtime to GPU.'

In [ ]:
# 5. Settings — edit as needed
TRIAXIAL = False        # True for triaxial, False for axisymmetric
N_PARTICLES = 10000
N_MIN = 3
VAR_RATIO = 9.0
N_BINS = 36
MIN_COUNT = 3
NUM_LIVE = 500
MAX_ITERATIONS = 50000
DLOGZ = 0.01
OUTPUT_DIR = 'results'

In [ ]:
# 6. Load data
import numpy as np
from main import load_stream_data

dict_data = load_stream_data(CSV_PATH, n_bins=N_BINS, min_count=MIN_COUNT)
print(f'Loaded {len(dict_data["theta"])} data points')

In [ ]:
# 7. Sanity check — compile & time a single likelihood call
import time as timer
import jax.numpy as jnp
from prior import logprior_axi, logprior_tri, sample_prior_axi, sample_prior_tri
from llikelihood import logl

jax_data = {k: jnp.array(v) for k, v in dict_data.items()}

# Draw a random sample from the prior
import jax.random as random
key = random.PRNGKey(0)
if TRIAXIAL:
    test_params = sample_prior_tri(key, 1)[0]
else:
    test_params = sample_prior_axi(key, 1)[0]

print(f'Test params shape: {test_params.shape}')
print(f'Test params: {test_params}')

# First call: JIT compilation
print('\n--- JIT compilation (first call) ---')
t0 = timer.time()
ll_val = logl(test_params, jax_data, N_PARTICLES, N_MIN, VAR_RATIO, TRIAXIAL)
jax.block_until_ready(ll_val)
t_compile = timer.time() - t0
print(f'logL = {float(ll_val):.2f}')
print(f'Time (with compilation): {t_compile:.2f}s')

# Second call: pure execution speed
print('\n--- Compiled execution (second call) ---')
t0 = timer.time()
ll_val = logl(test_params, jax_data, N_PARTICLES, N_MIN, VAR_RATIO, TRIAXIAL)
jax.block_until_ready(ll_val)
t_exec = timer.time() - t0
print(f'logL = {float(ll_val):.2f}')
print(f'Time (compiled): {t_exec:.4f}s')

# Average over multiple calls
print('\n--- Timing 10 calls ---')
t0 = timer.time()
for _ in range(10):
    ll_val = logl(test_params, jax_data, N_PARTICLES, N_MIN, VAR_RATIO, TRIAXIAL)
    jax.block_until_ready(ll_val)
t_avg = (timer.time() - t0) / 10
print(f'Average per call: {t_avg:.4f}s ({1/t_avg:.1f} calls/sec)')
print(f'\nDevice: {jax.devices()[0]}')

In [ ]:
# 8. Run BlackJAX nested sampling
import json
import pickle

from fit import blackjax_ns_fit

# Create output directory
run_name = f'nlive{NUM_LIVE}_npart{N_PARTICLES}_nmin{N_MIN}_vr{VAR_RATIO}'
run_dir = os.path.join(OUTPUT_DIR, run_name)
os.makedirs(run_dir, exist_ok=True)

# Save hyperparameters
hyperparams = {
    'num_live': NUM_LIVE, 'max_iterations': MAX_ITERATIONS,
    'dlogZ': DLOGZ, 'n_particles': N_PARTICLES,
    'n_min': N_MIN, 'var_ratio': VAR_RATIO,
    'n_bins': N_BINS, 'min_count': MIN_COUNT,
}
with open(os.path.join(run_dir, 'hyperparameters.json'), 'w') as f:
    json.dump(hyperparams, f, indent=2)

mode = 'triaxial' if TRIAXIAL else 'axisymmetric'
print(f'Fitting {mode} model with BlackJAX NS on {jax.devices()[0]}')
print(f'Run directory: {run_dir}')

dict_results = blackjax_ns_fit(
    dict_data, n_particles=N_PARTICLES, n_min=N_MIN,
    var_ratio=VAR_RATIO, num_live=NUM_LIVE,
    max_iterations=MAX_ITERATIONS, dlogZ_threshold=DLOGZ,
    triaxial=TRIAXIAL)

# Save results
with open(os.path.join(run_dir, 'dict_results.pkl'), 'wb') as f:
    pickle.dump(dict_results, f)
with open(os.path.join(run_dir, 'dict_data.pkl'), 'wb') as f:
    pickle.dump(dict_data, f)

print(f'\nResults saved to {run_dir}')
print(f'log Z = {dict_results["log_Z"]:.2f}')
print(f'ESS = {dict_results["ESS"]:.0f}')

In [ ]:
# 9. Corner plot
import corner
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 18})

LABELS_AXI = ['logM', 'Rs', 'dirx', 'diry', 'dirz',
              'logm', 'rs', 'x0', 'z0', 'vx0', 'vy0', 'vz0', 'time', 'sig']
LABELS_TRI = ['logM', 'Rs', 'p', 'q', 'dirx', 'diry', 'dirz',
              'logm', 'rs', 'x0', 'z0', 'vx0', 'vy0', 'vz0', 'time', 'sig']

labels = LABELS_TRI if TRIAXIAL else LABELS_AXI
fig_corner = corner.corner(dict_results['samps'], labels=labels, color='blue',
                           quantiles=[0.16, 0.5, 0.84], show_titles=True,
                           title_kwargs={'fontsize': 12})
fig_corner.savefig(os.path.join(run_dir, 'corner_plot.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# 10. Best fit plot
import StreaMAX
import jax.numpy as jnp
from utils import params_to_stream

best_params = dict_results['samps'][np.argmax(dict_results['logl'])]
theta_stream, r_stream, xv_stream = params_to_stream(best_params, N_PARTICLES, triaxial=TRIAXIAL)

r_bin, _, _ = jax.vmap(StreaMAX.get_track_2D, in_axes=(None, None, 0, None))(
    theta_stream, r_stream, dict_data['theta'], dict_data['bin_width'])

dt = dict_data['delta_theta']
c, s = np.cos(dt), np.sin(dt)
x0 = np.array(xv_stream[:, 0])
y0 = np.array(xv_stream[:, 1])
x_stream = x0 * c - y0 * s
y_stream = x0 * s + y0 * c
theta_abs = dict_data['theta'] + dt
x_data = dict_data['r'] * np.cos(theta_abs)
y_data = dict_data['r'] * np.sin(theta_abs)
r_bin_np = np.array(r_bin)
x_bin = r_bin_np * np.cos(theta_abs)
y_bin = r_bin_np * np.sin(theta_abs)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].scatter(x_stream, y_stream, c='blue', s=1, alpha=0.3, label='Best fit stream')
axes[0].scatter(x_bin, y_bin, c='lime', s=40, zorder=5, label='Model track')
axes[0].scatter(x_data, y_data, c='red', s=30, zorder=6, label='Data')
axes[0].set_xlabel('X (kpc)'); axes[0].set_ylabel('Y (kpc)')
axes[0].set_aspect('equal'); axes[0].legend(fontsize=12)

axes[1].errorbar(dict_data['theta'], dict_data['r'], yerr=dict_data['r_err'],
                 fmt='o', color='red', capsize=3, label='Data')
axes[1].scatter(dict_data['theta'], r_bin_np, c='lime', s=40, zorder=5, label='Model track')
axes[1].scatter(np.array(theta_stream), np.array(r_stream), c='blue', s=1, alpha=0.1)
axes[1].set_xlabel(r'$\theta$ (rad)'); axes[1].set_ylabel('r (kpc)')
axes[1].legend(fontsize=12)

fig.tight_layout()
fig.savefig(os.path.join(run_dir, 'best_fit.pdf'), bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 11. Flattening posterior
from utils import get_q

samps = dict_results['samps']

if TRIAXIAL:
    p_samps, q_samps = samps[:, 2], samps[:, 3]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(p_samps, bins=30, density=True, alpha=0.7, color='blue', range=(0.5, 1.0))
    axes[0].axvline(np.median(p_samps), color='blue', ls='--', lw=2)
    axes[0].axvline(1.0, color='k', lw=2)
    axes[0].set_xlabel('p = b/a'); axes[0].set_ylabel('Density')
    axes[1].hist(q_samps, bins=30, density=True, alpha=0.7, color='blue', range=(0.5, 1.0))
    axes[1].axvline(np.median(q_samps), color='blue', ls='--', lw=2)
    axes[1].axvline(1.0, color='k', lw=2)
    axes[1].set_xlabel('q = c/a'); axes[1].set_ylabel('Density')
    axes[2].scatter(p_samps, q_samps, s=1, alpha=0.2, color='blue')
    axes[2].plot([0.5, 1.0], [0.5, 1.0], 'k--', lw=1)
    axes[2].set_xlabel('p = b/a'); axes[2].set_ylabel('q = c/a')
    axes[2].set_xlim(0.5, 1.0); axes[2].set_ylim(0.5, 1.0); axes[2].set_aspect('equal')
else:
    q_samps = get_q(samps[:, 2], samps[:, 3], samps[:, 4])
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.hist(q_samps, bins=30, density=True, alpha=0.7, color='blue', range=(0.5, 1.5))
    ax.axvline(np.median(q_samps), color='blue', ls='--', lw=2)
    ax.axvline(1.0, color='k', lw=2)
    ax.set_xlabel('Halo Flattening q'); ax.set_ylabel('Density')

fig.tight_layout()
fig.savefig(os.path.join(run_dir, 'flattening.pdf'), bbox_inches='tight', dpi=150)
plt.show()